In [1]:
# ============================================================
# Notebook : 08_figures_tables.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : Assemble the final manuscript figures and tables from the cached
#            result CSVs: the controlled-ablation figure (AUC by variant with
#            error bars) and consolidated result tables. All figures are
#            grayscale, 600 dpi, saved as .png and .pdf, without captions.
# Note     : Anonymous repository for double-blind review.
# ============================================================

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns

ROOT    = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TAB_DIR = ROOT / "results" / "tables"; FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({"figure.dpi":150,"savefig.dpi":600,"savefig.bbox":"tight",
                     "font.family":"DejaVu Sans","axes.unicode_minus":False})
def savefig(fig, name):
    fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
GC = {"d":"#222222","m":"#777777","l":"#BBBBBB"}

# ── Ablation figure (AUC by variant, mean +/- std) ───────────
abl = pd.read_csv(TAB_DIR / "cv_ablation.csv")
order = ["Proposed: Causal + lag, all", "B: Causal + NO lag",
         "A: Fully-connected + lag", "A: Random graph + lag", "A: No inter-sensor + lag",
         "C: Causal + lag, EM-only", "C: Causal + lag, SM-only"]
order = [v for v in order if v in abl.variant.unique()]
means = [abl[abl.variant==v].auc_roc.mean() for v in order]
stds  = [abl[abl.variant==v].auc_roc.std()  for v in order]
shade = [GC["d"]] + [GC["m"]]*(len(order)-1)
fig, ax = plt.subplots(figsize=(8.5, 4))
xs = np.arange(len(order))
ax.bar(xs, means, yerr=stds, capsize=3, color=shade, edgecolor="black", linewidth=0.6)
ax.axhline(0.5, color="black", ls="--", lw=0.8)
ax.set_xticks(xs); ax.set_xticklabels([v.replace(": ",":\n").replace(" + ","\n+ ") for v in order], fontsize=7)
ax.set_ylabel("AUC-ROC"); ax.set_ylim(0, 1.0)
savefig(fig, "fig09_ablation_cv")

# ── Consolidated main results table ──────────────────────────
summ = pd.read_csv(TAB_DIR / "cv_summary.csv")
main = pd.DataFrame({
    "Model": summ["model"],
    "Macro F1": [f"{m:.3f} ± {s:.3f}" for m,s in zip(summ.macro_f1_mean, summ.macro_f1_std)],
    "Binary F1": [f"{m:.3f} ± {s:.3f}" for m,s in zip(summ.binary_f1_mean, summ.binary_f1_std)],
    "AUC-ROC": [f"{m:.3f} ± {s:.3f}" for m,s in zip(summ.auc_roc_mean, summ.auc_roc_std)],
})
main.to_csv(TAB_DIR / "table_main_results.csv", index=False)
print("Main results (5-fold mean ± std):")
print(main.to_string(index=False))

# ── Consolidated ablation table ──────────────────────────────
ab = (abl.groupby("variant")[["macro_f1","binary_f1","auc_roc"]].agg(["mean","std"]).round(3))
ab.columns = ["_".join(c) for c in ab.columns]
ab = ab.reindex(order).reset_index()
ab_out = pd.DataFrame({
    "Variant": ab["variant"],
    "Macro F1": [f"{m:.3f} ± {s:.3f}" for m,s in zip(ab.macro_f1_mean, ab.macro_f1_std)],
    "Binary F1": [f"{m:.3f} ± {s:.3f}" for m,s in zip(ab.binary_f1_mean, ab.binary_f1_std)],
    "AUC-ROC": [f"{m:.3f} ± {s:.3f}" for m,s in zip(ab.auc_roc_mean, ab.auc_roc_std)],
})
ab_out.to_csv(TAB_DIR / "table_ablation.csv", index=False)
print("\nAblation (5-fold mean ± std):")
print(ab_out.to_string(index=False))

# ── Inventory of all figures / tables ────────────────────────
print("\nFigures in results/figures:")
for p in sorted(FIG_DIR.glob("*.pdf")): print("  ", p.name)
print("Tables in results/tables:")
for p in sorted(TAB_DIR.glob("*.csv")): print("  ", p.name)
print("\n08_figures_tables complete.")


Main results (5-fold mean ± std):
                Model      Macro F1     Binary F1       AUC-ROC
Single-Modal MLP (EM) 0.578 ± 0.089 0.453 ± 0.086 0.678 ± 0.110
     Early Fusion MLP 0.664 ± 0.069 0.488 ± 0.132 0.738 ± 0.149
     Attention Fusion 0.675 ± 0.066 0.505 ± 0.126 0.751 ± 0.132
   Transformer Fusion 0.671 ± 0.074 0.501 ± 0.145 0.754 ± 0.147
Proposed (Causal+Lag) 0.622 ± 0.066 0.422 ± 0.139 0.709 ± 0.123

Ablation (5-fold mean ± std):
                    Variant      Macro F1     Binary F1       AUC-ROC
Proposed: Causal + lag, all 0.622 ± 0.066 0.422 ± 0.139 0.709 ± 0.123
         B: Causal + NO lag 0.620 ± 0.063 0.421 ± 0.137 0.706 ± 0.122
   A: Fully-connected + lag 0.597 ± 0.051 0.400 ± 0.118 0.690 ± 0.114
      A: Random graph + lag 0.600 ± 0.049 0.396 ± 0.124 0.676 ± 0.141
   A: No inter-sensor + lag 0.594 ± 0.049 0.391 ± 0.128 0.679 ± 0.123
   C: Causal + lag, EM-only 0.568 ± 0.073 0.414 ± 0.089 0.673 ± 0.098
   C: Causal + lag, SM-only 0.635 ± 0.073 0.423 ± 0.164 0.706